# 09 — Full-cohort selection and final model fit

This notebook repeats the locked training-only selection procedure once on all 1,040 primary patients and fits the final direct and two-stage model artifacts. It runs the same three stages as Notebooks 05–07 inside five stratified folds created with seed 42:

1. feature-pipeline screening (Notebook 05);
2. nine-family screening of the two retained pipelines (Notebook 06);
3. focused tuning of the two retained combinations (Notebook 07).

The selected direct, Stage 1, and Stage 2 pipelines are then refitted on all 1,040 patients and saved.

Before the full-cohort run, the notebook reruns seed 0's saved outer-training partition and must reproduce the saved Notebook 05, 06, and 07 decisions exactly. This confirms that the copied procedure is unchanged.

**Boundary:** these full-cohort fits are deployable and explanation artifacts. Every patient contributed to their selection and fitting, so their training-set predictions and selection scores are not performance estimates. Performance comes only from Notebook 07's 20 outer evaluations. No outer-test or sensitivity result is used here.

## 1. Check the modeling environment

Confirm that the external model packages and `cloudpickle` are available. `cloudpickle` saves each fitted pipeline together with this notebook's transformer definition, so the models reload in a fresh Python session. The CPU setting prevents a noisy joblib hardware-detection warning.

In [1]:
import importlib.util
import os

# Avoid noisy macOS physical-core detection in joblib while preserving the
# available logical-core limit for parallel estimators.
os.environ.setdefault(
    "LOKY_MAX_CPU_COUNT",
    str(max(1, (os.cpu_count() or 2) - 1)),
)

required_external_packages = ["xgboost", "lightgbm", "catboost", "cloudpickle"]
missing_packages = [
    package
    for package in required_external_packages
    if importlib.util.find_spec(package) is None
]
if missing_packages:
    raise ModuleNotFoundError(
        "Install the missing packages in this notebook environment, restart the "
        f"kernel, and rerun: {missing_packages}"
    )
print("External packages are available.")

External packages are available.


## 2. Import the analysis tools

Load the data, statistical, preprocessing, modeling, and serialization tools. As in Notebook 05, general `FutureWarning` messages are silenced; every model fit still records warnings through the checked fit helpers.

In [2]:
from pathlib import Path
import hashlib
import json
import pickle
import platform
import subprocess
import sys
import time
import warnings

import cloudpickle
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats
from sklearn import __version__ as sklearn_version
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import (
    ExtraTreesClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, recall_score
from sklearn.model_selection import StratifiedKFold
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVC, SVC
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier, __version__ as xgboost_version
from lightgbm import LGBMClassifier, __version__ as lightgbm_version
from catboost import CatBoostClassifier, __version__ as catboost_version

warnings.filterwarnings("ignore", category=FutureWarning)

## 3. Locate the frozen inputs

Use the primary 26-feature table, the Notebook 03 splits, the frozen Notebook 05–07 configuration manifests, and the saved Notebook 05–07 decisions used by the reproduction gate.

In [3]:
working_directory = Path.cwd().resolve()
project_root = next(
    (
        folder
        for folder in [working_directory, *working_directory.parents]
        if (folder / "AGENTS.md").is_file()
        and (folder / "docs/research_protocol.md").is_file()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError("Launch this notebook from within the project directory.")

final_run_directory = project_root / "results/final_pipeline/06_final_fit_and_performance_summary"
aggregation_directory = final_run_directory / "02_patient_level_aggregation"
split_directory = final_run_directory / "03_setup_and_splits"
feature_screen_directory = final_run_directory / "05_feature_pipeline_screening"
family_screen_directory = final_run_directory / "06_model_family_screening"
tuning_directory = final_run_directory / "07_tuning_and_outer_evaluation"
output_directory = final_run_directory / "09_full_cohort_fit"
reproduction_directory = output_directory / "reproduction_seed_00"
model_directory = output_directory / "models"
for folder in [output_directory, reproduction_directory, model_directory]:
    folder.mkdir(parents=True, exist_ok=True)

input_paths = {
    "predictors": aggregation_directory / "primary_1040_26_predictors.csv",
    "outcomes": aggregation_directory / "primary_1040_outcome_metadata.csv",
    "feature_manifest": split_directory / "primary_feature_manifest.csv",
    "outer_splits": split_directory / "outer_split_assignments.csv",
    "inner_folds": split_directory / "inner_fold_assignments.csv",
    "selector_manifest": feature_screen_directory / "selector_configuration_manifest.csv",
    "engineering_manifest": feature_screen_directory / "engineering_configuration_manifest.csv",
    "model_scope": family_screen_directory / "model_scope_manifest.csv",
    "tuning_grid": tuning_directory / "tuning_grid_manifest.csv",
    "saved_retained_pipelines": feature_screen_directory / "retained_feature_pipelines.csv",
    "saved_retained_combinations": family_screen_directory / "retained_for_focused_tuning.csv",
    "saved_tuned_winners": tuning_directory / "tuned_partition_winners.csv",
    "primary_outer_validation": tuning_directory / "outer_evaluation_validation.csv",
}
missing_inputs = [name for name, path in input_paths.items() if not path.is_file()]
assert not missing_inputs, f"Missing required inputs: {missing_inputs}"

print("Project root:", project_root)
print("Output directory:", output_directory.relative_to(project_root))

Project root: /Users/rafsan_temp/Library/CloudStorage/OneDrive-SeattleUniversity/SU Projects/pd-fall-risk
Output directory: results/final_pipeline/06_final_fit_and_performance_summary/09_full_cohort_fit


## 4. Load and validate the frozen inputs

Confirm the 1,040-patient input, the frozen configuration counts, and that Notebook 07's outer evaluation passed before the final artifacts are built.

In [4]:
predictors = pd.read_csv(
    input_paths["predictors"],
    dtype={"PATNO": "string"},
    low_memory=False,
).set_index("PATNO")
outcomes = pd.read_csv(
    input_paths["outcomes"],
    dtype={"PATNO": "string"},
    usecols=["PATNO", "falls_class"],
).set_index("PATNO")["falls_class"].astype(int)
feature_manifest = pd.read_csv(input_paths["feature_manifest"])
outer_assignments = pd.read_csv(input_paths["outer_splits"], dtype={"PATNO": "string"})
inner_assignments = pd.read_csv(input_paths["inner_folds"], dtype={"PATNO": "string"})
selector_manifest = pd.read_csv(input_paths["selector_manifest"])
engineering_manifest = pd.read_csv(input_paths["engineering_manifest"])
model_scope = pd.read_csv(input_paths["model_scope"])
tuning_grid = pd.read_csv(input_paths["tuning_grid"])
saved_retained_pipelines = pd.read_csv(input_paths["saved_retained_pipelines"])
saved_retained_combinations = pd.read_csv(input_paths["saved_retained_combinations"])
saved_tuned_winners = pd.read_csv(input_paths["saved_tuned_winners"])
primary_outer_validation = pd.read_csv(input_paths["primary_outer_validation"])

features = feature_manifest["feature"].tolist()
targets = ["direct", "stage_1", "stage_2"]

assert predictors.shape == (1040, 26) and predictors.columns.tolist() == features
assert set(predictors.index) == set(outcomes.index)
assert len(selector_manifest) == 8 and len(engineering_manifest) == 7
assert model_scope["candidate_winner"].astype(str).str.lower().eq("true").sum() == 9
assert len(tuning_grid) == 110
assert primary_outer_validation["passed"].astype(str).str.lower().eq("true").all()

print("Patients:", len(predictors))
print("Outcome classes:", outcomes.value_counts().sort_index().to_dict())
print("Frozen tuning grid rows:", len(tuning_grid))

Patients: 1040
Outcome classes: {0: 712, 1: 227, 2: 101}
Frozen tuning grid rows: 110


## 5. Define the two development partitions

The full-cohort folds apply Notebook 03's inner-fold rule to all 1,040 patients: order patients by numeric patient number, then create five stratified folds with shuffle seed 42. Every fold must contain all three outcome classes.

The reproduction partition is seed 0's saved outer-training set and its saved five inner folds. Random seeds inside every stage follow the Notebook 05–07 formulas, using 42 or 0 as the split seed. The final refit seed is `500000 + 42`, matching Notebook 07's refit rule.

In [5]:
FULL_COHORT_SEED = 42
REPRODUCTION_SEED = 0
FINAL_FIT_SEED = 500_000 + FULL_COHORT_SEED

ordered_ids = sorted(predictors.index, key=int)
full_cohort_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=FULL_COHORT_SEED)
full_cohort_folds = pd.DataFrame([
    {"PATNO": ordered_ids[position], "inner_validation_fold": fold}
    for fold, (_, validation_positions) in enumerate(
        full_cohort_cv.split(ordered_ids, outcomes.loc[ordered_ids].to_numpy())
    )
    for position in validation_positions
])
fold_balance = (
    full_cohort_folds.assign(falls_class=outcomes.loc[full_cohort_folds["PATNO"]].to_numpy())
    .groupby(["inner_validation_fold", "falls_class"])
    .size()
    .rename("patients")
    .reset_index()
)

reproduction_rows = outer_assignments.loc[outer_assignments["split_seed"].eq(REPRODUCTION_SEED)]
partitions = {
    REPRODUCTION_SEED: {
        "label": "seed 0 reproduction",
        "development_ids": sorted(reproduction_rows.loc[reproduction_rows["role"].eq("train"), "PATNO"]),
        "folds": inner_assignments.loc[
            inner_assignments["split_seed"].eq(REPRODUCTION_SEED),
            ["PATNO", "inner_validation_fold"],
        ],
        "directory": reproduction_directory,
    },
    FULL_COHORT_SEED: {
        "label": "full cohort seed 42",
        "development_ids": sorted(predictors.index),
        "folds": full_cohort_folds,
        "directory": output_directory,
    },
}

assert full_cohort_folds["PATNO"].is_unique and set(full_cohort_folds["PATNO"]) == set(predictors.index)
assert fold_balance.groupby("inner_validation_fold")["falls_class"].nunique().eq(3).all()
assert set(partitions[REPRODUCTION_SEED]["folds"]["PATNO"]) == set(partitions[REPRODUCTION_SEED]["development_ids"])
display(fold_balance.pivot(index="inner_validation_fold", columns="falls_class", values="patients"))
print("Reproduction development patients:", len(partitions[REPRODUCTION_SEED]["development_ids"]))

falls_class,0,1,2
inner_validation_fold,,,
0,143,45,20
1,143,45,20
2,142,45,21
3,142,46,20
4,142,46,20


Reproduction development patients: 728


# Part A — Locked procedure definitions

The next sections recreate the Notebook 05–07 definitions without methodological change. The screening configurations, model starting settings, and tuning grid are loaded from their frozen manifests rather than retyped.

## 6. Clinical preprocessing and representation groups

Declare the nominal, ordinal, and missing-state families, then derive missing-state indicators before filling and apply the Part IV structural-zero rule only when therapy is recorded as `No`.

In [6]:
NOMINAL_FEATURES = {
    "DXPOSINS", "DXRIGID", "DOPTHERST", "FEATPOSHYP",
    "ANYFAMPD", "DXTREMOR", "DXBRADY", "DOMSIDE",
}
ORDINAL_FEATURES = {
    "FRZGT12M", "SCAU14", "SCAU16", "NP1SLPD", "NP1URIN",
    "NP3GAIT_COMBINED_MAX", "NP3PSTBL_COMBINED_MAX", "NHY_COMBINED_MAX",
    "NP1CNST",
}
MISSING_INDICATORS = [
    "FOG_FORM_MISSING", "NQ_FORM_MISSING", "PART_IV_FORM_MISSING",
]


def representation_group(source):
    if source in {"FRZGT12M", "FOG_FORM_MISSING"}:
        return "GROUP_FREEZING_FORM"
    if source in {"NQ_GAUSSIAN_REVISION", "NQ_FORM_MISSING"}:
        return "GROUP_NEUROQOL_FORM"
    if source in {"NP4TOT", "PART_IV_FORM_MISSING"}:
        return "GROUP_PART_IV_FORM"
    return source


def clinical_frame(frame):
    raw = frame[features].copy()
    indicators = pd.DataFrame(index=raw.index)
    indicators["FOG_FORM_MISSING"] = raw["FRZGT12M"].isna().astype(int)
    indicators["NQ_FORM_MISSING"] = raw["NQ_GAUSSIAN_REVISION"].isna().astype(int)
    indicators["PART_IV_FORM_MISSING"] = raw["NP4TOT"].isna().astype(int)

    structural_zero = raw["NP4TOT"].isna() & raw["DOPTHERST"].eq("No")
    raw.loc[structural_zero, "NP4TOT"] = 0.0
    return pd.concat([raw, indicators], axis=1)


def clinical_raw(patient_ids):
    return clinical_frame(predictors.loc[list(patient_ids)])

## 7. Notebook 05 dense screening matrices

Fit numeric fills, nominal categories, one-hot encoding, and scaling on one training fold, then apply them unchanged to that fold's validation patients. The Stage A reference screens use these matrices.

In [7]:
def dense_pair(training_ids, validation_ids):
    raw_train = clinical_raw(training_ids)
    raw_validation = clinical_raw(validation_ids)

    nominal = [column for column in raw_train.columns if column in NOMINAL_FEATURES]
    numeric = [column for column in raw_train.columns if column not in nominal]
    train = raw_train.copy()
    validation = raw_validation.copy()

    freezing_mode = train["FRZGT12M"].mode(dropna=True)
    if freezing_mode.empty:
        raise ValueError("FRZGT12M has no observed value in this training fold.")
    train["FRZGT12M"] = train["FRZGT12M"].fillna(freezing_mode.iloc[0])
    validation["FRZGT12M"] = validation["FRZGT12M"].fillna(freezing_mode.iloc[0])

    median_columns = [column for column in numeric if column != "FRZGT12M"]
    medians = train[median_columns].median()
    if medians.isna().any():
        missing = medians[medians.isna()].index.tolist()
        raise ValueError(f"No training value available for: {missing}")
    train[median_columns] = train[median_columns].fillna(medians)
    validation[median_columns] = validation[median_columns].fillna(medians)

    for column in nominal:
        train[column] = train[column].astype("string").fillna("Missing").astype(str)
        validation[column] = (
            validation[column].astype("string").fillna("Missing").astype(str)
        )

    transformer = ColumnTransformer(
        [
            ("numeric", StandardScaler(), numeric),
            (
                "nominal",
                OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                nominal,
            ),
        ],
        verbose_feature_names_out=False,
    ).set_output(transform="pandas")

    X_train = transformer.fit_transform(train)
    X_validation = transformer.transform(validation)

    source_by_column = {column: column for column in numeric}
    encoder = transformer.named_transformers_["nominal"]
    encoded_names = encoder.get_feature_names_out(nominal)
    encoded_sources = [
        source
        for source, categories in zip(nominal, encoder.categories_)
        for _ in categories
    ]
    source_by_column.update(dict(zip(encoded_names, encoded_sources)))
    group_by_column = {
        column: representation_group(source_by_column[column])
        for column in X_train.columns
    }

    assert X_train.notna().all().all()
    assert X_validation.notna().all().all()
    assert X_train.columns.equals(X_validation.columns)

    return raw_train, X_train, X_validation, group_by_column

## 8. Targets, metrics, and warning handling

Build the direct, Stage 1, and Stage 2 targets; Stage 2 uses true fallers only and codes rare fall as class 0. The Stage A reference fits stop on convergence warnings as in Notebook 05. All Stage B and Stage C model fits, and the final fits, stop with one concise message on any warning, as in Notebooks 06 and 07.

In [8]:
def target_data(target_name, training_ids, validation_ids):
    y_train = outcomes.loc[list(training_ids)]
    y_validation = outcomes.loc[list(validation_ids)]
    if target_name == "direct":
        return list(training_ids), list(validation_ids), y_train, y_validation
    if target_name == "stage_1":
        return (
            list(training_ids),
            list(validation_ids),
            y_train.gt(0).astype(int),
            y_validation.gt(0).astype(int),
        )
    y_train = y_train[y_train.gt(0)]
    y_validation = y_validation[y_validation.gt(0)]
    return (
        y_train.index.tolist(),
        y_validation.index.tolist(),
        y_train.eq(2).astype(int),
        y_validation.eq(2).astype(int),
    )


def classification_metrics(target_name, truth, predictions):
    labels = [0, 1, 2] if target_name == "direct" else [0, 1]
    recalls = recall_score(
        truth,
        predictions,
        labels=labels,
        average=None,
        zero_division=0,
    )
    result = {
        "macro_f1": f1_score(truth, predictions, labels=labels, average="macro", zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(truth, predictions),
        "accuracy": accuracy_score(truth, predictions),
        "recall_class_0": recalls[0],
        "recall_class_1": recalls[1],
        "recall_class_2": recalls[2] if target_name == "direct" else np.nan,
    }
    result["priority_recall"] = (
        result["recall_class_0"]
        if target_name == "stage_2"
        else result["recall_class_1"]
    )
    return result


def fit_reference_checked(estimator, X, y, context):
    try:
        with warnings.catch_warnings():
            warnings.filterwarnings("error", category=ConvergenceWarning)
            estimator.fit(X, y)
    except ConvergenceWarning as error:
        raise RuntimeError(
            f"{context} did not converge. Increase max_iter before continuing."
        ) from error
    return estimator


def fit_checked(estimator, X, y, context, **fit_parameters):
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        estimator.fit(X, y, **fit_parameters)

    if caught:
        unique_messages = list(dict.fromkeys(str(item.message) for item in caught))
        convergence = any(
            issubclass(item.category, ConvergenceWarning)
            for item in caught
        )
        warning_type = "convergence warning" if convergence else "model warning"
        raise RuntimeError(
            f"{context} emitted a {warning_type}: {unique_messages[0]}"
        )
    return estimator

## 9. Corrected fold-specific statistical selector

Calculate p-values from observed training values only, apply Benjamini–Hochberg correction across that fold's representations, and keep declared representation groups together.

In [9]:
def bh_adjust(p_values):
    p_values = np.asarray(p_values, dtype=float)
    adjusted = np.full(len(p_values), np.nan)
    valid = np.flatnonzero(np.isfinite(p_values))
    if not len(valid):
        return adjusted
    order = valid[np.argsort(p_values[valid])]
    ranked = p_values[order] * len(valid) / np.arange(1, len(valid) + 1)
    ranked = np.minimum.accumulate(ranked[::-1])[::-1]
    adjusted[order] = np.minimum(ranked, 1.0)
    return adjusted


def numeric_p_value(values, target, ordinal):
    observed = pd.DataFrame({"value": values, "target": target}).dropna()
    groups = [
        observed.loc[observed["target"].eq(label), "value"].astype(float).to_numpy()
        for label in sorted(observed["target"].unique())
    ]
    if len(groups) < 2 or min(len(group) for group in groups) < 2:
        return np.nan
    if np.unique(np.concatenate(groups)).size < 2:
        return 1.0

    if ordinal:
        test = (
            stats.mannwhitneyu(*groups, alternative="two-sided")
            if len(groups) == 2
            else stats.kruskal(*groups)
        )
        return float(test.pvalue)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        normality = [
            stats.normaltest(group).pvalue
            if len(group) >= 8 and np.unique(group).size >= 3
            else 0.0
            for group in groups
        ]
        variance_p = stats.levene(*groups, center="median").pvalue
    parametric = (
        all(np.isfinite(normality))
        and min(normality) >= 0.05
        and np.isfinite(variance_p)
        and variance_p >= 0.05
    )
    if len(groups) == 2:
        test = (
            stats.ttest_ind(*groups, equal_var=True)
            if parametric
            else stats.mannwhitneyu(*groups, alternative="two-sided")
        )
    else:
        test = stats.f_oneway(*groups) if parametric else stats.kruskal(*groups)
    return float(test.pvalue)


def categorical_p_value(values, target, random_seed):
    categories = values.astype("string").fillna("Missing")
    table = pd.crosstab(categories, target)
    table = table.loc[table.sum(axis=1).gt(0)]
    if table.shape[0] < 2 or table.shape[1] < 2:
        return 1.0
    asymptotic = stats.chi2_contingency(table, correction=False)
    sparse = (
        asymptotic.expected_freq.min() < 1
        or (asymptotic.expected_freq < 5).mean() > 0.20
    )
    if not sparse:
        return float(asymptotic.pvalue)
    method = stats.PermutationMethod(
        n_resamples=999,
        rng=np.random.default_rng(random_seed),
    )
    return float(
        stats.chi2_contingency(
            table,
            correction=False,
            method=method,
        ).pvalue
    )


def select_groups_fdr(raw, target, random_seed):
    p_values = []
    for position, column in enumerate(raw.columns):
        if column in NOMINAL_FEATURES or column in MISSING_INDICATORS:
            p_value = categorical_p_value(
                raw[column],
                target,
                random_seed + position,
            )
        else:
            p_value = numeric_p_value(
                raw[column],
                target,
                ordinal=column in ORDINAL_FEATURES,
            )
        p_values.append(p_value)

    q_values = bh_adjust(p_values)
    selected_sources = [
        column
        for column, q_value in zip(raw.columns, q_values)
        if np.isfinite(q_value) and q_value <= 0.05
    ]
    if not selected_sources:
        finite = np.flatnonzero(np.isfinite(q_values))
        fallback = finite[np.argmin(q_values[finite])] if len(finite) else 0
        selected_sources = [raw.columns[fallback]]
    return {representation_group(source) for source in selected_sources}

## 10. Notebook 05 selectors and reference models

Apply the group-aware FDR, one-vs-rest `liblinear` L1, and Extra-Trees selectors to the Stage A dense matrices, and define the fixed balanced logistic and shallow Extra-Trees reference models.

In [10]:
def columns_for_groups(selected_groups, group_by_column):
    columns = [
        column
        for column, group in group_by_column.items()
        if group in selected_groups
    ]
    if not columns:
        raise RuntimeError("The selector returned no encoded columns.")
    return columns


def fdr_select(raw_train, target, group_by_column, random_seed):
    selected_groups = select_groups_fdr(raw_train, target, random_seed)
    return columns_for_groups(selected_groups, group_by_column), selected_groups


def l1_select(X_train, target, group_by_column, C_value, random_seed):
    base_estimator = LogisticRegression(
        solver="liblinear",
        l1_ratio=1.0,
        C=C_value,
        class_weight="balanced",
        max_iter=5000,
        random_state=random_seed,
    )
    selector = OneVsRestClassifier(base_estimator, n_jobs=-1)
    fit_reference_checked(selector, X_train, target, "L1 selector")
    coefficients = np.max(
        np.vstack([
            np.abs(estimator.coef_).reshape(-1)
            for estimator in selector.estimators_
        ]),
        axis=0,
    )
    selected_groups = {
        group_by_column[column]
        for column, coefficient in zip(X_train.columns, coefficients)
        if coefficient > 1e-10
    }
    if not selected_groups:
        strongest = X_train.columns[int(np.argmax(coefficients))]
        selected_groups = {group_by_column[strongest]}
    return columns_for_groups(selected_groups, group_by_column), selected_groups


def tree_select(X_train, target, group_by_column, threshold_multiplier, random_seed):
    selector = ExtraTreesClassifier(
        n_estimators=120,
        max_depth=6,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=random_seed,
        n_jobs=-1,
    )
    selector.fit(X_train, target)

    group_importance = {}
    for column, importance in zip(X_train.columns, selector.feature_importances_):
        group = group_by_column[column]
        group_importance[group] = group_importance.get(group, 0.0) + float(importance)

    threshold = np.median(list(group_importance.values())) * threshold_multiplier
    selected_groups = {
        group
        for group, importance in group_importance.items()
        if importance >= threshold
    }
    if not selected_groups:
        selected_groups = {max(group_importance, key=group_importance.get)}
    return columns_for_groups(selected_groups, group_by_column), selected_groups


def selected_feature_columns(
    selector_name, selector_parameter, raw_train, X_train, target, group_by_column, random_seed,
):
    if selector_name == "none":
        return X_train.columns.tolist(), set(group_by_column.values())
    if selector_name == "corrected_fdr":
        return fdr_select(raw_train, target, group_by_column, random_seed)
    if selector_name == "l1":
        C_value = float(selector_parameter.split("=")[1])
        return l1_select(X_train, target, group_by_column, C_value, random_seed)
    multiplier = 1.25 if "1.25" in selector_parameter else 1.0
    return tree_select(X_train, target, group_by_column, multiplier, random_seed)


def reference_model(model_name, random_seed):
    if model_name == "logistic":
        return LogisticRegression(
            C=1.0,
            class_weight="balanced",
            max_iter=3000,
            random_state=random_seed,
        )
    return ExtraTreesClassifier(
        n_estimators=120,
        max_depth=6,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=random_seed,
        n_jobs=-1,
    )

## 11. Feature engineering

Apply one interaction or PCA representation after fold-specific imputation and scaling. Interactions keep their main effects; PCA replaces only its declared source columns.

In [11]:
def engineered_pair(X_train, X_validation, branch_row):
    train = X_train.copy()
    validation = X_validation.copy()
    sources = branch_row["source_features"].split(" | ")

    if branch_row["branch_kind"] == "interaction":
        branch = branch_row["branch"]
        train[branch] = train[sources[0]].to_numpy() * train[sources[1]].to_numpy()
        validation[branch] = (
            validation[sources[0]].to_numpy()
            * validation[sources[1]].to_numpy()
        )
        return train, validation

    pca = PCA(
        n_components=float(branch_row["variance_threshold"]),
        svd_solver="full",
    )
    training_components = pca.fit_transform(train[sources])
    validation_components = pca.transform(validation[sources])
    component_names = [
        f"{branch_row['branch']}_PC{number + 1}"
        for number in range(training_components.shape[1])
    ]

    train = train.drop(columns=sources)
    validation = validation.drop(columns=sources)
    train[component_names] = training_components
    validation[component_names] = validation_components
    return train, validation


engineering_definitions = {
    row.branch: {
        "kind": row.branch_kind,
        "sources": row.source_features.split(" | "),
        "threshold": None if pd.isna(row.variance_threshold) else float(row.variance_threshold),
    }
    for row in engineering_manifest.itertuples(index=False)
}

## 12. Fold-fitted candidate transformer

This is the Notebook 06/07 transformer. It keeps preprocessing, interaction/PCA construction, and feature selection inside each fitted pipeline, so the saved final models carry every learned value with them. The longer cell is kept intact because these steps share fitted state.

In [12]:
class CandidateTransformer(BaseEstimator, TransformerMixin):
    def __init__(
        self,
        candidate_kind,
        selector,
        selector_parameter,
        branch=None,
        backend="scaled_dense",
        random_state=42,
    ):
        self.candidate_kind = candidate_kind
        self.selector = selector
        self.selector_parameter = selector_parameter
        self.branch = branch
        self.backend = backend
        self.random_state = random_state

    def _fit_preprocessing(self, raw):
        self.nominal_columns_ = [column for column in raw if column in NOMINAL_FEATURES]
        self.numeric_columns_ = [column for column in raw if column not in self.nominal_columns_]

        freezing_mode = raw["FRZGT12M"].mode(dropna=True)
        if freezing_mode.empty:
            raise ValueError("FRZGT12M has no observed training value.")
        self.freezing_fill_ = float(freezing_mode.iloc[0])

        median_columns = [column for column in self.numeric_columns_ if column != "FRZGT12M"]
        self.medians_ = raw[median_columns].median()
        if self.medians_.isna().any():
            missing = self.medians_[self.medians_.isna()].index.tolist()
            raise ValueError(f"No training value available for: {missing}")

        dense_numeric = raw[self.numeric_columns_].copy()
        dense_numeric["FRZGT12M"] = dense_numeric["FRZGT12M"].fillna(
            self.freezing_fill_
        )
        dense_numeric[median_columns] = dense_numeric[median_columns].fillna(
            self.medians_
        )
        self.scaler_ = StandardScaler().fit(dense_numeric)

        nominal = (
            raw[self.nominal_columns_]
            .astype("string")
            .fillna("Missing")
            .astype(str)
        )
        self.encoder_ = OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        ).fit(nominal)
        self.encoded_nominal_columns_ = self.encoder_.get_feature_names_out(
            self.nominal_columns_
        ).tolist()

        source_by_column = {column: column for column in self.numeric_columns_}
        encoded_sources = [
            source
            for source, categories in zip(
                self.nominal_columns_,
                self.encoder_.categories_,
            )
            for _ in categories
        ]
        source_by_column.update(
            dict(zip(self.encoded_nominal_columns_, encoded_sources))
        )
        self.group_by_column_ = {
            column: representation_group(source)
            for column, source in source_by_column.items()
        }

    def _matrices(self, raw):
        dense_numeric = raw[self.numeric_columns_].copy()
        dense_numeric["FRZGT12M"] = dense_numeric["FRZGT12M"].fillna(
            self.freezing_fill_
        )
        median_columns = [column for column in self.numeric_columns_ if column != "FRZGT12M"]
        dense_numeric[median_columns] = dense_numeric[median_columns].fillna(
            self.medians_
        )
        scaled_numeric = pd.DataFrame(
            self.scaler_.transform(dense_numeric),
            index=raw.index,
            columns=self.numeric_columns_,
        )

        nominal = (
            raw[self.nominal_columns_]
            .astype("string")
            .fillna("Missing")
            .astype(str)
        )
        encoded = pd.DataFrame(
            self.encoder_.transform(nominal),
            index=raw.index,
            columns=self.encoded_nominal_columns_,
        )
        native_numeric = raw[self.numeric_columns_].apply(
            pd.to_numeric,
            errors="coerce",
        )
        return {
            "scaled_dense": pd.concat([scaled_numeric, encoded], axis=1),
            "unscaled_dense": pd.concat([dense_numeric, encoded], axis=1),
            "native": pd.concat([native_numeric, encoded], axis=1),
        }

    def _apply_engineering(self, matrix, fit):
        if self.candidate_kind != "engineered_representation":
            return matrix
        definition = engineering_definitions[self.branch]
        matrix = matrix.copy()
        sources = definition["sources"]

        if definition["kind"] == "interaction":
            matrix[self.branch] = (
                matrix[sources[0]].to_numpy()
                * matrix[sources[1]].to_numpy()
            )
            return matrix

        if fit:
            self.pca_ = PCA(
                n_components=definition["threshold"],
                svd_solver="full",
            ).fit(matrix[sources])
        components = self.pca_.transform(matrix[sources])
        matrix = matrix.drop(columns=sources)
        names = [
            f"{self.branch}_PC{index + 1}"
            for index in range(components.shape[1])
        ]
        matrix[names] = components
        return matrix

    def _select_columns(self, raw, dense, target):
        if self.selector == "none":
            self.selected_groups_ = set(self.group_by_column_.values())
            return dense.columns.tolist()

        if self.selector == "corrected_fdr":
            selected_groups = select_groups_fdr(raw, target, self.random_state)
        elif self.selector == "l1":
            C_value = float(self.selector_parameter.split("=")[1])
            base_estimator = LogisticRegression(
                solver="liblinear",
                l1_ratio=1.0,
                C=C_value,
                class_weight="balanced",
                max_iter=5000,
                random_state=self.random_state,
            )
            selector = OneVsRestClassifier(base_estimator, n_jobs=-1)
            selector.fit(dense, target)
            importance = np.max(
                np.vstack([
                    np.abs(estimator.coef_).reshape(-1)
                    for estimator in selector.estimators_
                ]),
                axis=0,
            )
            selected_groups = {
                self.group_by_column_[column]
                for column, value in zip(dense.columns, importance)
                if value > 1e-10
            }
            if not selected_groups:
                strongest = dense.columns[int(np.argmax(importance))]
                selected_groups = {self.group_by_column_[strongest]}
        else:
            multiplier = 1.25 if "1.25" in self.selector_parameter else 1.0
            selector = ExtraTreesClassifier(
                n_estimators=120,
                max_depth=6,
                min_samples_leaf=5,
                class_weight="balanced",
                random_state=self.random_state,
                n_jobs=-1,
            )
            selector.fit(dense, target)
            group_importance = {}
            for column, value in zip(dense.columns, selector.feature_importances_):
                group = self.group_by_column_[column]
                group_importance[group] = group_importance.get(group, 0.0) + float(value)
            threshold = np.median(list(group_importance.values())) * multiplier
            selected_groups = {
                group
                for group, value in group_importance.items()
                if value >= threshold
            }
            if not selected_groups:
                selected_groups = {max(group_importance, key=group_importance.get)}

        self.selected_groups_ = selected_groups
        return [
            column
            for column in dense
            if self.group_by_column_[column] in selected_groups
        ]

    def fit(self, X, y):
        raw = clinical_frame(X)
        self._fit_preprocessing(raw)
        matrices = self._matrices(raw)
        dense = self._apply_engineering(matrices["scaled_dense"], fit=True)

        if self.candidate_kind == "engineered_representation":
            self.force_dense_ = True
            self.selected_columns_ = dense.columns.tolist()
            self.selected_groups_ = set(self.group_by_column_.values()) | {self.branch}
        else:
            self.force_dense_ = False
            self.selected_columns_ = self._select_columns(raw, dense, y)
        if not self.selected_columns_:
            raise RuntimeError("Candidate transformer selected no columns.")
        return self

    def transform(self, X):
        raw = clinical_frame(X)
        matrices = self._matrices(raw)
        if self.force_dense_:
            output = self._apply_engineering(
                matrices["scaled_dense"],
                fit=False,
            )
        else:
            output = matrices[self.backend]
        return output[self.selected_columns_]

    def get_feature_names_out(self, input_features=None):
        return np.asarray(self.selected_columns_, dtype=object)

## 13. Model families and frozen settings

Load the nine family starting settings from the Notebook 06 scope manifest and build each estimator exactly as Notebooks 06 and 07 did. RBF SVC receives probability estimates only in the final fit, as in Notebook 07's outer refits.

In [13]:
MODEL_FAMILIES = [
    "logistic", "linear_svc", "rbf_svc", "random_forest", "extra_trees",
    "hist_gradient_boosting", "xgboost", "lightgbm", "catboost",
]
MODEL_ORDER = {family: order for order, family in enumerate(MODEL_FAMILIES)}
REFERENCE_MODEL_ORDER = {"logistic": 0, "extra_trees": 1}
MODEL_BACKEND = {
    "logistic": "scaled_dense",
    "linear_svc": "scaled_dense",
    "rbf_svc": "scaled_dense",
    "random_forest": "unscaled_dense",
    "extra_trees": "unscaled_dense",
    "hist_gradient_boosting": "native",
    "xgboost": "native",
    "lightgbm": "native",
    "catboost": "native",
}
MODEL_STARTING_CONFIG = {
    row.model_family: json.loads(row.screening_parameters)
    for row in model_scope.itertuples(index=False)
    if row.model_family in MODEL_FAMILIES
}
assert list(MODEL_STARTING_CONFIG) == MODEL_FAMILIES
assert model_scope.set_index("model_family").loc[MODEL_FAMILIES, "backend"].to_dict() == MODEL_BACKEND
GRID_SIZES = tuning_grid.groupby("model_family").size()


def build_model(family, target_name, random_seed, parameters, probability=False):
    parameters = dict(parameters)
    if family == "logistic":
        return LogisticRegression(
            max_iter=5000,
            random_state=random_seed,
            **parameters,
        ), None
    if family == "linear_svc":
        return LinearSVC(
            dual="auto",
            max_iter=10000,
            random_state=random_seed,
            **parameters,
        ), None
    if family == "rbf_svc":
        return SVC(
            cache_size=1000,
            probability=probability,
            random_state=random_seed,
            **parameters,
        ), None
    if family == "random_forest":
        return RandomForestClassifier(
            random_state=random_seed,
            n_jobs=-1,
            **parameters,
        ), None
    if family == "extra_trees":
        return ExtraTreesClassifier(
            random_state=random_seed,
            n_jobs=-1,
            **parameters,
        ), None
    if family == "hist_gradient_boosting":
        return HistGradientBoostingClassifier(
            random_state=random_seed,
            **parameters,
        ), None
    if family == "xgboost":
        weight_mode = parameters.pop("sample_weight")
        class_count = 3 if target_name == "direct" else 2
        parameters.update({
            "objective": "multi:softprob" if class_count == 3 else "binary:logistic",
            "eval_metric": "mlogloss" if class_count == 3 else "logloss",
            "random_state": random_seed,
            "n_jobs": -1,
            "verbosity": 0,
        })
        if class_count == 3:
            parameters["num_class"] = 3
        return XGBClassifier(**parameters), weight_mode
    if family == "lightgbm":
        return LGBMClassifier(
            random_state=random_seed,
            n_jobs=-1,
            verbosity=-1,
            **parameters,
        ), None

    loss = "MultiClass" if target_name == "direct" else "Logloss"
    return CatBoostClassifier(
        random_seed=random_seed,
        loss_function=loss,
        verbose=False,
        allow_writing_files=False,
        thread_count=-1,
        **parameters,
    ), None


def fitted_pipeline(
    row, family, target_name, random_seed, training_ids, y_train, parameters, context,
    probability=False,
):
    branch = None if pd.isna(row.branch) else row.branch
    estimator, weight_mode = build_model(
        family,
        target_name,
        random_seed,
        parameters,
        probability=probability,
    )
    pipeline = Pipeline([
        ("candidate", CandidateTransformer(
            candidate_kind=row.candidate_kind,
            selector=row.selector,
            selector_parameter=row.selector_parameter,
            branch=branch,
            backend=MODEL_BACKEND[family],
            random_state=random_seed,
        )),
        ("model", estimator),
    ])
    fit_parameters = {}
    if weight_mode == "balanced":
        fit_parameters["model__sample_weight"] = compute_sample_weight("balanced", y_train)
    fit_checked(pipeline, predictors.loc[training_ids], y_train, context=context, **fit_parameters)
    return pipeline

# Part B — Stage runners

Each runner performs one Notebook 05–07 stage for one development partition. Every learned step is refitted inside the four training folds and scored on the held-out fold.

## 14. Stage A: feature-pipeline screen

For each target and fold, evaluate the eight selector/reference configurations and compare each of the ten engineering branches with the unengineered Extra-Trees baseline (Notebook 05).

In [14]:
def fold_ids(development_ids, fold_table, inner_fold):
    validation_ids = sorted(
        fold_table.loc[fold_table["inner_validation_fold"].eq(inner_fold), "PATNO"].tolist()
    )
    training_ids = sorted(set(development_ids) - set(validation_ids))
    assert set(validation_ids).issubset(development_ids)
    return training_ids, validation_ids


def run_feature_screen(split_seed, development_ids, fold_table):
    selector_rows = []
    engineering_rows = []
    for target_name in targets:
        for inner_fold in range(5):
            training_ids, validation_ids = fold_ids(development_ids, fold_table, inner_fold)
            train_ids, valid_ids, y_train, y_validation = target_data(
                target_name, training_ids, validation_ids,
            )

            random_seed = 100_000 + split_seed * 10 + inner_fold
            raw_train, X_train, X_validation, group_by_column = dense_pair(train_ids, valid_ids)
            selected_cache = {}
            for configuration in selector_manifest.itertuples(index=False):
                cache_key = (configuration.selector, configuration.selector_parameter)
                if cache_key not in selected_cache:
                    selected_cache[cache_key] = selected_feature_columns(
                        configuration.selector,
                        configuration.selector_parameter,
                        raw_train,
                        X_train,
                        y_train,
                        group_by_column,
                        random_seed,
                    )
                selected_columns, selected_groups = selected_cache[cache_key]
                model = reference_model(configuration.reference_model, random_seed)
                fit_start = time.perf_counter()
                fit_reference_checked(
                    model,
                    X_train[selected_columns],
                    y_train,
                    f"{configuration.reference_model} reference model",
                )
                predictions = model.predict(X_validation[selected_columns])
                selector_rows.append({
                    "split_seed": split_seed,
                    "target": target_name,
                    "inner_validation_fold": inner_fold,
                    "configuration_id": configuration.configuration_id,
                    "pipeline_key": configuration.pipeline_key,
                    "candidate_kind": configuration.candidate_kind,
                    "feature_set": configuration.feature_set,
                    "selector": configuration.selector,
                    "selector_parameter": configuration.selector_parameter,
                    "reference_model": configuration.reference_model,
                    "training_patients": len(y_train),
                    "validation_patients": len(y_validation),
                    "processed_input_columns": X_train.shape[1],
                    "selected_columns": len(selected_columns),
                    "selected_groups": len(selected_groups),
                    "selected_group_names": " | ".join(sorted(selected_groups)),
                    "selected_column_names": " | ".join(selected_columns),
                    "fit_seconds": time.perf_counter() - fit_start,
                    **classification_metrics(target_name, y_validation, predictions),
                })

            random_seed = 200_000 + split_seed * 10 + inner_fold
            _, baseline_train, baseline_validation, _ = dense_pair(train_ids, valid_ids)
            baseline_model = reference_model("extra_trees", random_seed)
            fit_reference_checked(
                baseline_model, baseline_train, y_train, "Extra-Trees engineering baseline",
            )
            baseline_scores = classification_metrics(
                target_name, y_validation, baseline_model.predict(baseline_validation),
            )
            for branch_row in engineering_manifest.to_dict("records"):
                X_branch_train, X_branch_validation = engineered_pair(
                    baseline_train, baseline_validation, branch_row,
                )
                model = reference_model("extra_trees", random_seed)
                fit_start = time.perf_counter()
                fit_reference_checked(
                    model,
                    X_branch_train,
                    y_train,
                    f"Extra-Trees engineering branch {branch_row['branch']}",
                )
                candidate_scores = classification_metrics(
                    target_name, y_validation, model.predict(X_branch_validation),
                )
                engineering_rows.append({
                    "split_seed": split_seed,
                    "target": target_name,
                    "inner_validation_fold": inner_fold,
                    "configuration_id": branch_row["configuration_id"],
                    "pipeline_key": branch_row["pipeline_key"],
                    "candidate_kind": branch_row["candidate_kind"],
                    "feature_set": branch_row["feature_set"],
                    "selector": branch_row["selector"],
                    "selector_parameter": branch_row["selector_parameter"],
                    "branch": branch_row["branch"],
                    "branch_kind": branch_row["branch_kind"],
                    "reference_model": branch_row["reference_model"],
                    "training_patients": len(y_train),
                    "validation_patients": len(y_validation),
                    "selected_columns": X_branch_train.shape[1],
                    "fit_seconds": time.perf_counter() - fit_start,
                    "baseline_macro_f1": baseline_scores["macro_f1"],
                    "baseline_priority_recall": baseline_scores["priority_recall"],
                    **candidate_scores,
                    "delta_macro_f1": candidate_scores["macro_f1"] - baseline_scores["macro_f1"],
                    "delta_priority_recall": (
                        candidate_scores["priority_recall"] - baseline_scores["priority_recall"]
                    ),
                })
        print(f"  Stage A complete for {target_name}", flush=True)
    return pd.DataFrame(selector_rows), pd.DataFrame(engineering_rows)

## 15. Stage A retention rule

An engineering branch is eligible only if it improves mean macro F1 over the baseline, improves at least three of five folds, and loses no more than 0.01 target-priority recall. Two distinct feature pipelines are retained per target by mean macro F1, then priority recall within 0.01, then fewer groups and columns (Notebook 05).

In [15]:
def retain_two_pipelines(group):
    remaining = group.loc[group["eligible_to_advance"]].copy()
    retained = []
    for rank in [1, 2]:
        if remaining.empty:
            raise RuntimeError("Fewer than two eligible feature pipelines remain.")
        best_macro_f1 = remaining["mean_macro_f1"].max()
        near_ties = remaining.loc[remaining["mean_macro_f1"].ge(best_macro_f1 - 0.01)].copy()
        near_ties["reference_model_order"] = near_ties["reference_model"].map(REFERENCE_MODEL_ORDER)
        chosen = near_ties.sort_values(
            [
                "mean_priority_recall", "mean_selected_groups",
                "mean_selected_columns", "reference_model_order",
                "pipeline_key", "configuration_id",
            ],
            ascending=[False, True, True, True, True, True],
        ).iloc[0]
        chosen_row = chosen.drop(labels="reference_model_order").to_dict()
        chosen_row["retained_rank"] = rank
        retained.append(chosen_row)
        remaining = remaining.loc[remaining["pipeline_key"].ne(chosen["pipeline_key"])]
    return pd.DataFrame(retained)


def retain_feature_pipelines(selector_folds, engineering_folds):
    selector_summary = selector_folds.groupby(
        [
            "split_seed", "target", "configuration_id", "pipeline_key",
            "candidate_kind", "feature_set", "selector",
            "selector_parameter", "reference_model",
        ],
        as_index=False,
    ).agg(
        mean_macro_f1=("macro_f1", "mean"),
        sd_macro_f1=("macro_f1", "std"),
        mean_priority_recall=("priority_recall", "mean"),
        mean_balanced_accuracy=("balanced_accuracy", "mean"),
        mean_accuracy=("accuracy", "mean"),
        mean_selected_columns=("selected_columns", "mean"),
        mean_selected_groups=("selected_groups", "mean"),
        inner_folds=("inner_validation_fold", "nunique"),
    )
    engineering_summary = engineering_folds.groupby(
        [
            "split_seed", "target", "configuration_id", "pipeline_key",
            "candidate_kind", "feature_set", "selector",
            "selector_parameter", "branch", "branch_kind", "reference_model",
        ],
        as_index=False,
    ).agg(
        mean_baseline_macro_f1=("baseline_macro_f1", "mean"),
        mean_macro_f1=("macro_f1", "mean"),
        sd_macro_f1=("macro_f1", "std"),
        mean_delta_macro_f1=("delta_macro_f1", "mean"),
        improving_folds=("delta_macro_f1", lambda values: int((values > 0).sum())),
        mean_priority_recall=("priority_recall", "mean"),
        mean_delta_priority_recall=("delta_priority_recall", "mean"),
        mean_balanced_accuracy=("balanced_accuracy", "mean"),
        mean_accuracy=("accuracy", "mean"),
        mean_selected_columns=("selected_columns", "mean"),
        inner_folds=("inner_validation_fold", "nunique"),
    )
    engineering_summary["mean_selected_groups"] = engineering_summary["mean_selected_columns"]
    engineering_summary["eligible_to_advance"] = (
        engineering_summary["mean_delta_macro_f1"].gt(0)
        & engineering_summary["improving_folds"].ge(3)
        & engineering_summary["mean_delta_priority_recall"].ge(-0.01)
    )

    selector_pool = selector_summary.copy()
    selector_pool["branch"] = pd.NA
    selector_pool["branch_kind"] = "raw"
    selector_pool["mean_delta_macro_f1"] = np.nan
    selector_pool["improving_folds"] = np.nan
    selector_pool["mean_delta_priority_recall"] = np.nan
    selector_pool["eligible_to_advance"] = True
    pool_columns = [
        "split_seed", "target", "configuration_id", "pipeline_key",
        "candidate_kind", "feature_set", "selector", "selector_parameter",
        "branch", "branch_kind", "reference_model", "mean_macro_f1",
        "sd_macro_f1", "mean_priority_recall", "mean_balanced_accuracy",
        "mean_accuracy", "mean_selected_columns", "mean_selected_groups",
        "inner_folds", "mean_delta_macro_f1", "improving_folds",
        "mean_delta_priority_recall", "eligible_to_advance",
    ]
    screening_pool = pd.concat(
        [selector_pool[pool_columns], engineering_summary[pool_columns]],
        ignore_index=True,
    )
    retained = pd.concat(
        [
            retain_two_pipelines(group)
            for _, group in screening_pool.groupby(["split_seed", "target"], sort=False)
        ],
        ignore_index=True,
    ).sort_values(["split_seed", "target", "retained_rank"])
    return selector_summary, engineering_summary, screening_pool, retained

## 16. Stage B: nine-family screen

Fit every family's frozen starting configuration on both retained feature pipelines in the same five folds, with a prior-class dummy benchmark. Retain the two best pipeline/family combinations per target using the same macro-F1, priority-recall, parsimony, and family-order rule (Notebook 06).

In [16]:
def run_family_screen(split_seed, development_ids, fold_table, retained_pipelines):
    rows = []
    dummy_rows = []
    for target_name in targets:
        candidate_rows = retained_pipelines.loc[retained_pipelines["target"].eq(target_name)]
        for inner_fold in range(5):
            training_ids, validation_ids = fold_ids(development_ids, fold_table, inner_fold)
            train_ids, valid_ids, y_train, y_validation = target_data(
                target_name, training_ids, validation_ids,
            )
            random_seed = 300_000 + split_seed * 10 + inner_fold

            dummy = DummyClassifier(strategy="prior")
            dummy.fit(np.zeros((len(y_train), 1)), y_train)
            dummy_rows.append({
                "split_seed": split_seed,
                "target": target_name,
                "inner_validation_fold": inner_fold,
                "training_patients": len(y_train),
                "validation_patients": len(y_validation),
                **classification_metrics(
                    target_name, y_validation, dummy.predict(np.zeros((len(y_validation), 1))),
                ),
            })

            for candidate_row in candidate_rows.itertuples(index=False):
                branch = None if pd.isna(candidate_row.branch) else candidate_row.branch
                for family in MODEL_FAMILIES:
                    fit_start = time.perf_counter()
                    pipeline = fitted_pipeline(
                        candidate_row,
                        family,
                        target_name,
                        random_seed,
                        train_ids,
                        y_train,
                        MODEL_STARTING_CONFIG[family],
                        context=(
                            f"seed {split_seed}, {target_name}, fold {inner_fold}, "
                            f"{candidate_row.pipeline_key}, {family}"
                        ),
                    )
                    predictions = pipeline.predict(predictors.loc[valid_ids])
                    fitted_transformer = pipeline.named_steps["candidate"]
                    rows.append({
                        "split_seed": split_seed,
                        "target": target_name,
                        "inner_validation_fold": inner_fold,
                        "retained_feature_rank": candidate_row.retained_rank,
                        "configuration_id": candidate_row.configuration_id,
                        "pipeline_key": candidate_row.pipeline_key,
                        "candidate_kind": candidate_row.candidate_kind,
                        "selector": candidate_row.selector,
                        "selector_parameter": candidate_row.selector_parameter,
                        "branch": branch,
                        "branch_kind": candidate_row.branch_kind,
                        "model_family": family,
                        "backend": (
                            "scaled_dense" if fitted_transformer.force_dense_ else MODEL_BACKEND[family]
                        ),
                        "training_patients": len(y_train),
                        "validation_patients": len(y_validation),
                        "selected_columns": len(fitted_transformer.selected_columns_),
                        "selected_groups": len(fitted_transformer.selected_groups_),
                        "selected_column_names": " | ".join(fitted_transformer.selected_columns_),
                        "fit_seconds": time.perf_counter() - fit_start,
                        **classification_metrics(target_name, y_validation, predictions),
                    })
        print(f"  Stage B complete for {target_name}", flush=True)
    return pd.DataFrame(rows), pd.DataFrame(dummy_rows)


def retain_two_combinations(group):
    remaining = group.copy()
    retained = []
    for rank in [1, 2]:
        best_macro_f1 = remaining["mean_macro_f1"].max()
        near_ties = remaining.loc[remaining["mean_macro_f1"].ge(best_macro_f1 - 0.01)].copy()
        near_ties["model_order"] = near_ties["model_family"].map(MODEL_ORDER)
        chosen = near_ties.sort_values(
            [
                "mean_priority_recall", "mean_selected_groups",
                "mean_selected_columns", "model_order",
                "pipeline_key", "model_family",
            ],
            ascending=[False, True, True, True, True, True],
        ).iloc[0]
        chosen_row = chosen.drop(labels="model_order").to_dict()
        chosen_row["retained_rank"] = rank
        retained.append(chosen_row)
        remaining = remaining.loc[
            ~(
                remaining["pipeline_key"].eq(chosen["pipeline_key"])
                & remaining["model_family"].eq(chosen["model_family"])
            )
        ]
    return pd.DataFrame(retained)


def retain_combinations(family_folds, dummy_folds):
    family_summary = family_folds.groupby(
        [
            "split_seed", "target", "configuration_id", "pipeline_key",
            "candidate_kind", "selector", "selector_parameter",
            "branch", "branch_kind", "model_family", "backend",
        ],
        dropna=False,
        as_index=False,
    ).agg(
        mean_macro_f1=("macro_f1", "mean"),
        sd_macro_f1=("macro_f1", "std"),
        mean_priority_recall=("priority_recall", "mean"),
        mean_balanced_accuracy=("balanced_accuracy", "mean"),
        mean_accuracy=("accuracy", "mean"),
        mean_selected_columns=("selected_columns", "mean"),
        mean_selected_groups=("selected_groups", "mean"),
        mean_fit_seconds=("fit_seconds", "mean"),
        inner_folds=("inner_validation_fold", "nunique"),
    )
    retained = pd.concat(
        [
            retain_two_combinations(group)
            for _, group in family_summary.groupby(["split_seed", "target"], sort=False)
        ],
        ignore_index=True,
    ).sort_values(["split_seed", "target", "retained_rank"])
    dummy_summary = dummy_folds.groupby(["split_seed", "target"], as_index=False).agg(
        mean_macro_f1=("macro_f1", "mean"),
        mean_priority_recall=("priority_recall", "mean"),
        mean_balanced_accuracy=("balanced_accuracy", "mean"),
        mean_accuracy=("accuracy", "mean"),
    )
    return family_summary, dummy_summary, retained

## 17. Stage C: focused tuning

Evaluate every frozen grid row for the two retained combinations in the same five folds, then choose one winner per target by mean macro F1, target-priority recall within 0.01, fewer groups and columns, and the prespecified simpler family (Notebook 07).

In [17]:
def run_focused_tuning(split_seed, development_ids, fold_table, retained_combinations):
    rows = []
    ordered = retained_combinations.sort_values(["split_seed", "target", "retained_rank"])
    for retained_row in ordered.itertuples(index=False):
        family_grid = tuning_grid.loc[
            tuning_grid["model_family"].eq(retained_row.model_family)
        ].sort_values("grid_id")
        branch = None if pd.isna(retained_row.branch) else retained_row.branch
        for grid_row in family_grid.itertuples(index=False):
            parameters = json.loads(grid_row.parameters)
            for inner_fold in range(5):
                training_ids, validation_ids = fold_ids(development_ids, fold_table, inner_fold)
                train_ids, valid_ids, y_train, y_validation = target_data(
                    retained_row.target, training_ids, validation_ids,
                )
                random_seed = 400_000 + split_seed * 10 + inner_fold
                fit_start = time.perf_counter()
                pipeline = fitted_pipeline(
                    retained_row,
                    retained_row.model_family,
                    retained_row.target,
                    random_seed,
                    train_ids,
                    y_train,
                    parameters,
                    context=(
                        f"seed {split_seed}, {retained_row.target}, rank {retained_row.retained_rank}, "
                        f"grid {grid_row.grid_id}, fold {inner_fold}"
                    ),
                )
                predictions = pipeline.predict(predictors.loc[valid_ids])
                fitted_transformer = pipeline.named_steps["candidate"]
                rows.append({
                    "split_seed": split_seed,
                    "target": retained_row.target,
                    "retained_rank": int(retained_row.retained_rank),
                    "configuration_id": retained_row.configuration_id,
                    "pipeline_key": retained_row.pipeline_key,
                    "candidate_kind": retained_row.candidate_kind,
                    "selector": retained_row.selector,
                    "selector_parameter": retained_row.selector_parameter,
                    "branch": branch,
                    "branch_kind": retained_row.branch_kind,
                    "model_family": retained_row.model_family,
                    "backend": (
                        "scaled_dense"
                        if fitted_transformer.force_dense_
                        else MODEL_BACKEND[retained_row.model_family]
                    ),
                    "grid_id": int(grid_row.grid_id),
                    "parameters": grid_row.parameters,
                    "inner_validation_fold": inner_fold,
                    "training_patients": len(y_train),
                    "validation_patients": len(y_validation),
                    "selected_columns": len(fitted_transformer.selected_columns_),
                    "selected_groups": len(fitted_transformer.selected_groups_),
                    "selected_column_names": " | ".join(fitted_transformer.selected_columns_),
                    "fit_seconds": time.perf_counter() - fit_start,
                    **classification_metrics(retained_row.target, y_validation, predictions),
                })
        print(
            f"  Stage C complete for {retained_row.target}, rank {retained_row.retained_rank} "
            f"({retained_row.model_family}, {len(family_grid)} grid rows)",
            flush=True,
        )
    return pd.DataFrame(rows)


def choose_winner(group):
    best_macro_f1 = group["mean_macro_f1"].max()
    near_ties = group.loc[group["mean_macro_f1"].ge(best_macro_f1 - 0.01)].copy()
    near_ties["model_order"] = near_ties["model_family"].map(MODEL_ORDER)
    return near_ties.sort_values(
        [
            "mean_priority_recall", "mean_selected_groups", "mean_selected_columns",
            "model_order", "grid_id", "configuration_id",
        ],
        ascending=[False, True, True, True, True, True],
    ).iloc[0].drop(labels="model_order")


def choose_tuned_winners(tuning_folds):
    tuning_summary = tuning_folds.groupby(
        [
            "split_seed", "target", "retained_rank", "configuration_id", "pipeline_key",
            "candidate_kind", "selector", "selector_parameter", "branch", "branch_kind",
            "model_family", "backend", "grid_id", "parameters",
        ],
        dropna=False,
        as_index=False,
    ).agg(
        mean_macro_f1=("macro_f1", "mean"),
        sd_macro_f1=("macro_f1", "std"),
        mean_priority_recall=("priority_recall", "mean"),
        mean_balanced_accuracy=("balanced_accuracy", "mean"),
        mean_accuracy=("accuracy", "mean"),
        mean_selected_columns=("selected_columns", "mean"),
        mean_selected_groups=("selected_groups", "mean"),
        mean_fit_seconds=("fit_seconds", "mean"),
        inner_folds=("inner_validation_fold", "nunique"),
    )
    winners = pd.DataFrame([
        choose_winner(group)
        for _, group in tuning_summary.groupby(["split_seed", "target"], sort=False)
    ]).reset_index(drop=True)
    return tuning_summary, winners

## 18. Lock the run identity

Hash every upstream input, the full-cohort fold assignment, and the seed rules. Existing checkpoints can resume only when this configuration is unchanged.

In [18]:
RUN_VERSION = "final-notebook-09-full-cohort-fit-v2-26-features-d28"


def file_digest(file_path):
    digest = hashlib.sha256()
    with open(file_path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def frame_digest(frame):
    canonical = frame.to_csv(index=False, lineterminator="\n")
    return hashlib.sha256(canonical.encode()).hexdigest()


identity = {
    "run_version": RUN_VERSION,
    "input_sha256": {name: file_digest(path) for name, path in input_paths.items()},
    "full_cohort_folds_sha256": frame_digest(full_cohort_folds),
    "full_cohort_seed": FULL_COHORT_SEED,
    "reproduction_seed": REPRODUCTION_SEED,
    "final_fit_seed": FINAL_FIT_SEED,
    "stage_seed_rules": "Notebook 05: 100000/200000; Notebook 06: 300000; Notebook 07: 400000 + split_seed * 10 + fold",
    "selection_metric": "mean inner-fold macro F1",
    "near_tie_margin": 0.01,
}
configuration_hash = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()
manifest_path = output_directory / "run_manifest.json"
manifest = {
    **identity,
    "configuration_hash": configuration_hash,
    "python": platform.python_version(),
    "scikit_learn": sklearn_version,
    "xgboost": xgboost_version,
    "lightgbm": lightgbm_version,
    "catboost": catboost_version,
    "cloudpickle": cloudpickle.__version__,
}
if manifest_path.is_file():
    existing = json.loads(manifest_path.read_text())
    if existing.get("configuration_hash") != configuration_hash:
        raise RuntimeError(
            "Existing checkpoints belong to a different configuration. "
            "Preserve them and use a new output directory."
        )
else:
    manifest_path.write_text(json.dumps(manifest, indent=2) + "\n")

print("Configuration hash:", configuration_hash[:16])
print(pd.Series({
    name: manifest[name]
    for name in ["python", "scikit_learn", "xgboost", "lightgbm", "catboost", "cloudpickle"]
}).to_string())

Configuration hash: 66a9a9a77e08bcec
python          3.13.12
scikit_learn      1.8.0
xgboost           3.2.0
lightgbm          4.7.0
catboost         1.2.10
cloudpickle       3.1.2


## 19. Define the checkpointed partition run

Run Stages A–C in order for one partition. Each stage's fold results and retained candidates are written atomically and then read back, so a fresh run and a resumed run feed the next stage identically, as the separate Notebook 05–07 files did.

In [19]:
def atomic_csv(frame, file_path):
    temporary = file_path.with_suffix(file_path.suffix + ".tmp")
    frame.to_csv(temporary, index=False)
    temporary.replace(file_path)


def cached_frames(directory, filenames, compute):
    file_paths = [directory / filename for filename in filenames]
    if not all(file_path.is_file() for file_path in file_paths):
        for file_path, frame in zip(file_paths, compute()):
            atomic_csv(frame, file_path)
    return [pd.read_csv(file_path, low_memory=False) for file_path in file_paths]


def run_partition(split_seed):
    partition = partitions[split_seed]
    directory = partition["directory"]
    development_ids = partition["development_ids"]
    fold_table = partition["folds"]
    start = time.perf_counter()
    print(f"Running {partition['label']} ({len(development_ids)} patients)", flush=True)

    selector_folds, engineering_folds = cached_frames(
        directory,
        ["selector_inner_fold_results.csv", "engineering_inner_fold_results.csv"],
        lambda: run_feature_screen(split_seed, development_ids, fold_table),
    )
    selector_summary, engineering_summary, screening_pool, retained = retain_feature_pipelines(
        selector_folds, engineering_folds,
    )
    (retained_pipelines,) = cached_frames(
        directory, ["retained_feature_pipelines.csv"], lambda: [retained],
    )
    print(f"Stage A done after {(time.perf_counter() - start) / 60:.1f} min", flush=True)

    family_folds, dummy_folds = cached_frames(
        directory,
        ["family_screen_inner_fold_results.csv", "dummy_benchmark_inner_fold_results.csv"],
        lambda: run_family_screen(split_seed, development_ids, fold_table, retained_pipelines),
    )
    family_summary, dummy_summary, combinations = retain_combinations(family_folds, dummy_folds)
    (retained_combinations,) = cached_frames(
        directory, ["retained_for_focused_tuning.csv"], lambda: [combinations],
    )
    print(f"Stage B done after {(time.perf_counter() - start) / 60:.1f} min", flush=True)

    (tuning_folds,) = cached_frames(
        directory,
        ["focused_tuning_inner_fold_results.csv"],
        lambda: [run_focused_tuning(split_seed, development_ids, fold_table, retained_combinations)],
    )
    tuning_summary, winners = choose_tuned_winners(tuning_folds)
    print(f"Stage C done after {(time.perf_counter() - start) / 60:.1f} min", flush=True)

    return {
        "selector_folds": selector_folds,
        "engineering_folds": engineering_folds,
        "selector_summary": selector_summary,
        "engineering_summary": engineering_summary,
        "screening_pool": screening_pool,
        "retained_pipelines": retained_pipelines,
        "family_folds": family_folds,
        "dummy_folds": dummy_folds,
        "family_summary": family_summary,
        "dummy_summary": dummy_summary,
        "retained_combinations": retained_combinations,
        "tuning_folds": tuning_folds,
        "tuning_summary": tuning_summary,
        "winners": winners,
    }

# Part C — Reproduction gate

## 20. Rerun seed 0's outer-training partition

Run Stages A–C on seed 0's 728 outer-training patients with its saved inner folds. This takes a few minutes; progress is printed after each target and stage.

In [20]:
reproduction = run_partition(REPRODUCTION_SEED)

Running seed 0 reproduction (728 patients)
  Stage A complete for direct
  Stage A complete for stage_1
  Stage A complete for stage_2
Stage A done after 0.5 min
  Stage B complete for direct
  Stage B complete for stage_1
  Stage B complete for stage_2
Stage B done after 1.6 min
  Stage C complete for direct, rank 1 (catboost, 16 grid rows)
  Stage C complete for direct, rank 2 (catboost, 16 grid rows)
  Stage C complete for stage_1, rank 1 (rbf_svc, 18 grid rows)
  Stage C complete for stage_1, rank 2 (rbf_svc, 18 grid rows)
  Stage C complete for stage_2, rank 1 (lightgbm, 16 grid rows)
  Stage C complete for stage_2, rank 2 (random_forest, 8 grid rows)
Stage C done after 3.6 min


## 21. Confirm the saved seed 0 decisions are reproduced

The retained feature pipelines (Notebook 05), retained combinations (Notebook 06), and tuned winners (Notebook 07) must match the saved seed 0 decisions exactly, with the same mean macro F1. The full-cohort run starts only if every stage passes.

In [21]:
decision_specs = [
    (
        "Notebook 05 retained feature pipelines",
        reproduction["retained_pipelines"],
        saved_retained_pipelines,
        ["target", "retained_rank"],
        ["configuration_id", "pipeline_key"],
    ),
    (
        "Notebook 06 retained combinations",
        reproduction["retained_combinations"],
        saved_retained_combinations,
        ["target", "retained_rank"],
        ["pipeline_key", "model_family"],
    ),
    (
        "Notebook 07 tuned winners",
        reproduction["winners"],
        saved_tuned_winners,
        ["target"],
        ["configuration_id", "model_family", "grid_id"],
    ),
]

gate_rows = []
for stage, current, saved, keys, decisions in decision_specs:
    saved_seed = saved.loc[saved["split_seed"].eq(REPRODUCTION_SEED)]
    merged = current.merge(saved_seed, on=keys, suffixes=("_current", "_saved"), validate="one_to_one")
    same_decisions = len(merged) == len(saved_seed) == len(current) and all(
        merged[f"{column}_current"].astype(str).eq(merged[f"{column}_saved"].astype(str)).all()
        for column in decisions
    )
    score_difference = float(
        (merged["mean_macro_f1_current"] - merged["mean_macro_f1_saved"]).abs().max()
    )
    gate_rows.append({
        "stage": stage,
        "compared_rows": len(merged),
        "same_decisions": same_decisions,
        "max_macro_f1_difference": score_difference,
        "passed": same_decisions and score_difference <= 1e-9,
    })
reproduction_check = pd.DataFrame(gate_rows)
display(reproduction_check)
assert reproduction_check["passed"].all(), reproduction_check.loc[~reproduction_check["passed"]]
print("Seed 0 decisions reproduced exactly; the full-cohort run may start.")

,stage,compared_rows,same_decisions,max_macro_f1_difference,passed
0,Notebook 05 retained feature pipelines,6,True,0.0,True
1,Notebook 06 retained combinations,6,True,0.0,True
2,Notebook 07 tuned winners,3,True,0.0,True


Seed 0 decisions reproduced exactly; the full-cohort run may start.


# Part D — Full-cohort selection

## 22. Run the locked procedure on all 1,040 patients

Run Stages A–C on the full cohort with the seed-42 folds. This takes a few minutes. The selection scores printed below are optimistic because the same patients selected the pipeline; they are not performance estimates.

In [22]:
full_cohort = run_partition(FULL_COHORT_SEED)
full_cohort_winners = full_cohort["winners"]

display(full_cohort["retained_pipelines"][["target", "retained_rank", "pipeline_key", "mean_macro_f1"]])
display(full_cohort["retained_combinations"][["target", "retained_rank", "pipeline_key", "model_family", "mean_macro_f1"]])
display(full_cohort_winners[[
    "target", "pipeline_key", "model_family", "grid_id", "parameters",
    "mean_macro_f1", "mean_priority_recall",
]].rename(columns={
    "mean_macro_f1": "selection_macro_f1",
    "mean_priority_recall": "selection_priority_recall",
}))

Running full cohort seed 42 (1040 patients)
  Stage A complete for direct
  Stage A complete for stage_1
  Stage A complete for stage_2
Stage A done after 0.4 min
  Stage B complete for direct
  Stage B complete for stage_1
  Stage B complete for stage_2
Stage B done after 1.8 min
  Stage C complete for direct, rank 1 (catboost, 16 grid rows)
  Stage C complete for direct, rank 2 (catboost, 16 grid rows)
  Stage C complete for stage_1, rank 1 (extra_trees, 8 grid rows)
  Stage C complete for stage_1, rank 2 (hist_gradient_boosting, 16 grid rows)
  Stage C complete for stage_2, rank 1 (random_forest, 8 grid rows)
  Stage C complete for stage_2, rank 2 (extra_trees, 8 grid rows)
Stage C done after 5.1 min


,target,retained_rank,pipeline_key,mean_macro_f1
0,direct,1,raw__corrected_fdr__q<=0.05,0.522116
1,direct,2,raw__none__all,0.526037
2,stage_1,1,raw__corrected_fdr__q<=0.05,0.691229
3,stage_1,2,engineering__INT_DURATION_X_MOTOR,0.683817
4,stage_2,1,raw__extra_trees__threshold=median,0.695639
5,stage_2,2,raw__none__all,0.682109


,target,retained_rank,pipeline_key,model_family,mean_macro_f1
0,direct,1,raw__none__all,catboost,0.542285
1,direct,2,raw__corrected_fdr__q<=0.05,catboost,0.522944
2,stage_1,1,raw__corrected_fdr__q<=0.05,extra_trees,0.683112
3,stage_1,2,raw__corrected_fdr__q<=0.05,hist_gradient_boosting,0.690081
4,stage_2,1,raw__extra_trees__threshold=median,random_forest,0.685894
5,stage_2,2,raw__extra_trees__threshold=median,extra_trees,0.694445


,target,pipeline_key,model_family,grid_id,parameters,selection_macro_f1,selection_priority_recall
0,direct,raw__corrected_fdr__q<=0.05,catboost,5,"{""auto_class_weights"": ""Balanced"", ""depth"": 6,...",0.531912,0.462802
1,stage_1,raw__corrected_fdr__q<=0.05,extra_trees,5,"{""class_weight"": ""balanced"", ""max_depth"": null...",0.685798,0.649371
2,stage_2,raw__extra_trees__threshold=median,random_forest,0,"{""class_weight"": null, ""max_depth"": null, ""min...",0.686314,0.885894


# Part E — Final full-cohort models

## 23. Fit the three selected pipelines on all 1,040 patients

Refit the direct, Stage 1, and Stage 2 winners on every eligible patient using refit seed `500042`. Stage 2 trains only on the true fallers. RBF SVC fits probability estimates, as in Notebook 07.

In [23]:
all_ids = sorted(predictors.index)
final_pipelines = {}
final_training_ids = {}
for winner in full_cohort_winners.itertuples(index=False):
    y_all = outcomes.loc[all_ids]
    training_ids = list(all_ids)
    if winner.target == "direct":
        y_train = y_all
    elif winner.target == "stage_1":
        y_train = y_all.gt(0).astype(int)
    else:
        y_train = y_all[y_all.gt(0)].eq(2).astype(int)
        training_ids = y_train.index.tolist()

    final_pipelines[winner.target] = fitted_pipeline(
        winner,
        winner.model_family,
        winner.target,
        FINAL_FIT_SEED,
        training_ids,
        y_train,
        json.loads(winner.parameters),
        context=f"full-cohort {winner.target} fit",
        probability=winner.model_family == "rbf_svc",
    )
    final_training_ids[winner.target] = training_ids

print({target: len(ids) for target, ids in final_training_ids.items()})

{'direct': 1040, 'stage_1': 1040, 'stage_2': 328}


## 24. Record the fitted feature pipelines and fill values

List the columns and representation groups each final pipeline retained, and the training-derived placeholder values it applies to missing inputs. Dense models use these fills; native-missing models keep numeric gaps. Filled values are placeholders, not measured patient facts.

In [24]:
selected_rows = []
fill_rows = []
for winner in full_cohort_winners.itertuples(index=False):
    transformer = final_pipelines[winner.target].named_steps["candidate"]
    model_input = "scaled_dense" if transformer.force_dense_ else MODEL_BACKEND[winner.model_family]
    selected_rows.append({
        "target": winner.target,
        "configuration_id": winner.configuration_id,
        "pipeline_key": winner.pipeline_key,
        "candidate_kind": winner.candidate_kind,
        "selector": winner.selector,
        "selector_parameter": winner.selector_parameter,
        "branch": None if pd.isna(winner.branch) else winner.branch,
        "model_family": winner.model_family,
        "grid_id": int(winner.grid_id),
        "parameters": winner.parameters,
        "selection_macro_f1": winner.mean_macro_f1,
        "selection_priority_recall": winner.mean_priority_recall,
        "training_patients": len(final_training_ids[winner.target]),
        "model_input": model_input,
        "selected_group_count": len(transformer.selected_groups_),
        "selected_groups": " | ".join(sorted(transformer.selected_groups_)),
        "selected_column_count": len(transformer.selected_columns_),
        "selected_column_names": " | ".join(transformer.selected_columns_),
    })

    training_raw = clinical_frame(predictors.loc[final_training_ids[winner.target]])
    gap_columns = [
        column for column in transformer.numeric_columns_
        if training_raw[column].isna().any()
    ]
    for column in gap_columns:
        fill_rows.append({
            "target": winner.target,
            "feature": column,
            "rule": "training mode" if column == "FRZGT12M" else "training median",
            "fill_value": (
                transformer.freezing_fill_ if column == "FRZGT12M" else float(transformer.medians_[column])
            ),
            "missing_training_patients": int(training_raw[column].isna().sum()),
            "applies_to_model_input": model_input != "native",
        })

final_selected_pipelines = pd.DataFrame(selected_rows)
final_fill_values = pd.DataFrame(fill_rows)
display(final_selected_pipelines[[
    "target", "model_family", "model_input", "selected_group_count", "selected_column_count",
]])
display(final_fill_values)

,target,model_family,model_input,selected_group_count,selected_column_count
0,direct,catboost,native,20,29
1,stage_1,extra_trees,unscaled_dense,20,29
2,stage_2,random_forest,unscaled_dense,13,25


,target,feature,rule,fill_value,missing_training_patients,applies_to_model_input
0,direct,FRZGT12M,training mode,0.000000,63,False
1,direct,NQ_GAUSSIAN_REVISION,training median,1.600637,64,False
2,direct,NP4TOT,training median,2.000000,81,False
3,stage_1,FRZGT12M,training mode,0.000000,63,True
4,stage_1,NQ_GAUSSIAN_REVISION,training median,1.600637,64,True
5,stage_1,NP4TOT,training median,2.000000,81,True
6,stage_2,FRZGT12M,training mode,0.000000,28,True
7,stage_2,NQ_GAUSSIAN_REVISION,training median,2.631652,29,True
8,stage_2,NP4TOT,training median,5.000000,12,True


## 25. Define direct and hard-routed two-stage prediction

Use Notebook 07's score alignment: model probabilities where available, otherwise a fixed sigmoid/softmax of decision scores. The two-stage system predicts no fall when Stage 1 predicts no fall; otherwise Stage 2 assigns rare or recurrent fall. These helpers are applied to the cohort only to check the saved artifacts, not to measure performance.

In [25]:
def aligned_score_matrix(pipeline, frame, labels):
    model = pipeline.named_steps["model"]
    if hasattr(model, "predict_proba"):
        raw = np.asarray(pipeline.predict_proba(frame), dtype=float)
    else:
        decision = np.asarray(pipeline.decision_function(frame), dtype=float)
        if decision.ndim == 1:
            positive = 1.0 / (1.0 + np.exp(-np.clip(decision, -700, 700)))
            raw = np.column_stack([1.0 - positive, positive])
        else:
            shifted = decision - decision.max(axis=1, keepdims=True)
            exponentiated = np.exp(shifted)
            raw = exponentiated / exponentiated.sum(axis=1, keepdims=True)

    aligned = np.zeros((len(raw), len(labels)), dtype=float)
    classes = np.asarray(model.classes_).astype(int)
    for source_column, label in enumerate(classes):
        aligned[:, labels.index(int(label))] = raw[:, source_column]
    assert np.isfinite(aligned).all()
    assert np.allclose(aligned.sum(axis=1), 1.0)
    return aligned


def predict_classes(pipeline, frame):
    return np.asarray(pipeline.predict(frame)).astype(int).reshape(-1)


def predict_two_stage(stage_1_pipeline, stage_2_pipeline, frame):
    stage_1_scores = aligned_score_matrix(stage_1_pipeline, frame, [0, 1])
    stage_2_scores = aligned_score_matrix(stage_2_pipeline, frame, [0, 1])
    classes = np.where(
        predict_classes(stage_1_pipeline, frame) == 0,
        0,
        predict_classes(stage_2_pipeline, frame) + 1,
    )
    scores = np.column_stack([
        stage_1_scores[:, 0],
        stage_1_scores[:, 1] * stage_2_scores[:, 0],
        stage_1_scores[:, 1] * stage_2_scores[:, 1],
    ])
    return classes, scores


reference_frame = predictors.loc[all_ids]
in_memory_predictions = {
    target: predict_classes(pipeline, reference_frame)
    for target, pipeline in final_pipelines.items()
}
direct_scores = aligned_score_matrix(final_pipelines["direct"], reference_frame, [0, 1, 2])
two_stage_classes, two_stage_scores = predict_two_stage(
    final_pipelines["stage_1"], final_pipelines["stage_2"], reference_frame,
)
print("Direct and two-stage scores are finite and sum to one:", bool(
    np.allclose(direct_scores.sum(axis=1), 1.0) and np.allclose(two_stage_scores.sum(axis=1), 1.0)
))

Direct and two-stage scores are finite and sum to one: True


## 26. Save the three fitted pipelines

Save each pipeline with `cloudpickle`, which stores this notebook's transformer and helper definitions by value. An existing model file is never overwritten: it is reloaded and must give identical predictions to the newly fitted pipeline.

In [26]:
model_paths = {target: model_directory / f"{target}_pipeline.pkl" for target in targets}


def save_or_verify_model(pipeline, file_path):
    if file_path.is_file():
        with open(file_path, "rb") as handle:
            existing = pickle.load(handle)
        if not np.array_equal(
            predict_classes(existing, reference_frame),
            predict_classes(pipeline, reference_frame),
        ):
            raise FileExistsError(
                f"Existing model truly differs and was not overwritten: {file_path.name}"
            )
        return "already equivalent"
    temporary = file_path.with_suffix(file_path.suffix + ".tmp")
    with open(temporary, "wb") as handle:
        cloudpickle.dump(pipeline, handle)
    temporary.replace(file_path)
    return "created"


model_save_status = pd.DataFrame([
    {
        "target": target,
        "file": model_paths[target].name,
        "status": save_or_verify_model(final_pipelines[target], model_paths[target]),
    }
    for target in targets
])
model_save_status["sha256"] = [file_digest(model_paths[target]) for target in targets]
display(model_save_status)

,target,file,status,sha256
0,direct,direct_pipeline.pkl,created,7584c8f94e50de44ae58bf825670db13484c8ec05e93a8...
1,stage_1,stage_1_pipeline.pkl,created,4fbf8d985b5723db0c550ac97fefe533f5e7753405ff04...
2,stage_2,stage_2_pipeline.pkl,created,b863a19014fa648825ea18d1712e8d96a28dae27e3d738...


## 27. Reload the saved models in a fresh Python process

Load each saved file in a separate Python process that has none of this notebook's definitions, predict the 1,040 patients from the raw predictor CSV, and require identical predictions. This confirms that later interpretation notebooks can use the files directly.

In [27]:
reload_script = """
import json
import pickle
import sys

import numpy as np
import pandas as pd

frame = pd.read_csv(sys.argv[1], dtype={"PATNO": "string"}, low_memory=False)
frame = frame.set_index("PATNO").sort_index()
predictions = {}
for name, file_path in zip(sys.argv[2::2], sys.argv[3::2]):
    with open(file_path, "rb") as handle:
        model = pickle.load(handle)
    predictions[name] = np.asarray(model.predict(frame)).astype(int).reshape(-1).tolist()
print(json.dumps({"patients": frame.index.tolist(), "predictions": predictions}))
"""
arguments = [sys.executable, "-c", reload_script, str(input_paths["predictors"])]
for target in targets:
    arguments += [target, str(model_paths[target])]
completed = subprocess.run(
    arguments,
    capture_output=True,
    text=True,
    env={**os.environ, "PYTHONWARNINGS": "ignore"},
)
if completed.returncode != 0:
    raise RuntimeError(f"Fresh-process reload failed:\n{completed.stderr[-2000:]}")
reloaded = json.loads(completed.stdout.strip().splitlines()[-1])

model_reload_check = pd.DataFrame([
    {
        "target": target,
        "patients": len(reloaded["predictions"][target]),
        "same_patient_order": reloaded["patients"] == all_ids,
        "identical_predictions": np.array_equal(
            np.asarray(reloaded["predictions"][target]), in_memory_predictions[target],
        ),
    }
    for target in targets
])
display(model_reload_check)
assert model_reload_check[["same_patient_order", "identical_predictions"]].all().all()

,target,patients,same_patient_order,identical_predictions
0,direct,1040,True,True
1,stage_1,1040,True,True
2,stage_2,1040,True,True


## 28. Write the model manifest

Describe each saved file, its class coding, the two-stage routing rule, the required input columns, the seeds, and the package versions needed to reload it. An existing manifest must match exactly.

In [28]:
class_coding = {
    "direct": {"0": "no fall", "1": "rare fall", "2": "recurrent fall"},
    "stage_1": {"0": "no fall", "1": "any fall"},
    "stage_2": {"0": "rare fall", "1": "recurrent fall"},
}
model_manifest = {
    "run_version": RUN_VERSION,
    "configuration_hash": configuration_hash,
    "cohort": "primary 1,040-patient landmark cohort",
    "input_columns": features,
    "selection_seed": FULL_COHORT_SEED,
    "final_fit_seed": FINAL_FIT_SEED,
    "models": {
        row.target: {
            "file": model_paths[row.target].name,
            "sha256": file_digest(model_paths[row.target]),
            "model_family": row.model_family,
            "pipeline_key": row.pipeline_key,
            "parameters": json.loads(row.parameters),
            "training_patients": int(row.training_patients),
            "class_coding": class_coding[row.target],
        }
        for row in final_selected_pipelines.itertuples(index=False)
    },
    "two_stage_rule": (
        "Predict no fall when Stage 1 predicts class 0; otherwise the final class is "
        "Stage 2's class + 1 (1 = rare fall, 2 = recurrent fall)."
    ),
    "loading": "pickle.load with cloudpickle and the package versions below installed",
    "data_governance": (
        "Model files can contain derived patient-level training values (for example, "
        "RBF SVC support vectors); share them only under PPMI data-use terms."
    ),
    "performance_note": (
        "Full-cohort training-set predictions and selection scores are not performance "
        "estimates; report Notebook 07's outer evaluation."
    ),
    "package_versions": {
        name: manifest[name]
        for name in ["python", "scikit_learn", "xgboost", "lightgbm", "catboost", "cloudpickle"]
    },
}
model_manifest_path = model_directory / "model_manifest.json"
if model_manifest_path.is_file():
    if json.loads(model_manifest_path.read_text()) != json.loads(json.dumps(model_manifest)):
        raise FileExistsError("Existing model manifest truly differs and was not overwritten.")
    manifest_status = "already equivalent"
else:
    model_manifest_path.write_text(json.dumps(model_manifest, indent=2) + "\n")
    manifest_status = "created"
print("Model manifest:", manifest_status)

Model manifest: created


## 29. Validate the complete notebook

Confirm the reproduction gate, full-cohort fold coverage, complete stage coverage, one frozen-grid winner per target, correct final training sets, valid two-stage scores, and successful fresh-process reloading.

In [29]:
expected_tuning_rows = int(sum(
    GRID_SIZES.loc[family] * 5 for family in full_cohort["retained_combinations"]["model_family"]
))
faller_count = int(outcomes.gt(0).sum())
winner_grid_matches = full_cohort_winners.merge(
    tuning_grid, on=["model_family", "grid_id", "parameters"], how="inner",
)

full_cohort_validation = pd.DataFrame([
    {
        "check": "seed 0 reproduction gate passed",
        "passed": reproduction_check["passed"].all(),
        "detail": "Notebook 05, 06, and 07 decisions reproduced",
    },
    {
        "check": "full-cohort folds cover every patient once with all classes",
        "passed": full_cohort_folds["PATNO"].is_unique
        and len(full_cohort_folds) == 1040
        and fold_balance.groupby("inner_validation_fold")["falls_class"].nunique().eq(3).all(),
        "detail": "five stratified folds, seed 42",
    },
    {
        "check": "Stage A complete",
        "passed": len(full_cohort["selector_folds"]) == 3 * 8 * 5
        and len(full_cohort["engineering_folds"]) == 3 * len(engineering_manifest) * 5,
        "detail": "8 selector and 7 engineering configurations × 5 folds × 3 targets",
    },
    {
        "check": "two distinct feature pipelines retained per target",
        "passed": full_cohort["retained_pipelines"].groupby("target")["pipeline_key"].nunique().eq(2).all()
        and len(full_cohort["retained_pipelines"]) == 6,
        "detail": "Notebook 05 retention rule",
    },
    {
        "check": "Stage B complete",
        "passed": len(full_cohort["family_folds"]) == 3 * 2 * 9 * 5
        and set(full_cohort["family_folds"]["model_family"]) == set(MODEL_FAMILIES)
        and len(full_cohort["dummy_folds"]) == 15,
        "detail": "2 pipelines × 9 families × 5 folds × 3 targets",
    },
    {
        "check": "two combinations retained per target",
        "passed": full_cohort["retained_combinations"].groupby("target").size().eq(2).all()
        and len(full_cohort["retained_combinations"]) == 6,
        "detail": "Notebook 06 retention rule",
    },
    {
        "check": "Stage C complete",
        "passed": len(full_cohort["tuning_folds"]) == expected_tuning_rows,
        "detail": f"{expected_tuning_rows} frozen-grid fold fits",
    },
    {
        "check": "one frozen-grid winner per target",
        "passed": len(full_cohort_winners) == 3
        and set(full_cohort_winners["target"]) == set(targets)
        and len(winner_grid_matches) == 3,
        "detail": "Notebook 07 selection rule",
    },
    {
        "check": "final training sets are correct",
        "passed": len(final_training_ids["direct"]) == 1040
        and len(final_training_ids["stage_1"]) == 1040
        and len(final_training_ids["stage_2"]) == faller_count,
        "detail": f"Stage 2 uses the {faller_count} true fallers",
    },
    {
        "check": "all final pipelines retained columns",
        "passed": final_selected_pipelines["selected_column_count"].gt(0).all(),
        "detail": "no empty model input",
    },
    {
        "check": "direct and two-stage scores are valid",
        "passed": bool(
            np.isfinite(direct_scores).all()
            and np.isfinite(two_stage_scores).all()
            and np.allclose(direct_scores.sum(axis=1), 1.0)
            and np.allclose(two_stage_scores.sum(axis=1), 1.0)
        ),
        "detail": "hard-routed two-stage composition",
    },
    {
        "check": "saved models reload identically in a fresh process",
        "passed": model_reload_check[["same_patient_order", "identical_predictions"]].all().all(),
        "detail": "direct, Stage 1, and Stage 2 files",
    },
    {
        "check": "run identity still matches",
        "passed": json.loads(manifest_path.read_text())["configuration_hash"] == configuration_hash,
        "detail": "same frozen inputs, folds, and seeds",
    },
])
display(full_cohort_validation)
assert full_cohort_validation["passed"].all(), full_cohort_validation.loc[
    ~full_cohort_validation["passed"]
]
print(
    f"Full-cohort validation passed: {full_cohort_validation['passed'].sum()}/"
    f"{len(full_cohort_validation)}"
)

,check,passed,detail
0,seed 0 reproduction gate passed,True,"Notebook 05, 06, and 07 decisions reproduced"
1,full-cohort folds cover every patient once wit...,True,"five stratified folds, seed 42"
2,Stage A complete,True,8 selector and 7 engineering configurations × ...
3,two distinct feature pipelines retained per ta...,True,Notebook 05 retention rule
4,Stage B complete,True,2 pipelines × 9 families × 5 folds × 3 targets
5,two combinations retained per target,True,Notebook 06 retention rule
6,Stage C complete,True,360 frozen-grid fold fits
7,one frozen-grid winner per target,True,Notebook 07 selection rule
8,final training sets are correct,True,Stage 2 uses the 328 true fallers
9,all final pipelines retained columns,True,no empty model input


Full-cohort validation passed: 13/13


## 30. Save the full-cohort artifacts

Save the fold assignment, reproduction check, stage summaries, winners, final pipeline descriptions, fill values, reload check, and validation table. The stage fold results and retained candidates were already written by the checkpoints. Existing differing artifacts are never overwritten.

In [30]:
def frames_equivalent(current, existing):
    if current.columns.tolist() != existing.columns.tolist() or current.shape != existing.shape:
        return False
    for column in current.columns:
        left = current[column]
        right = existing[column]
        left_numeric = pd.to_numeric(left, errors="coerce")
        right_numeric = pd.to_numeric(right, errors="coerce")
        left_numeric_ok = left_numeric.notna().eq(left.notna()).all()
        right_numeric_ok = right_numeric.notna().eq(right.notna()).all()
        if left_numeric_ok and right_numeric_ok:
            if not np.allclose(
                left_numeric.to_numpy(dtype=float),
                right_numeric.to_numpy(dtype=float),
                rtol=1e-12,
                atol=1e-12,
                equal_nan=True,
            ):
                return False
        else:
            left_text = left.astype("string").fillna("<NA>").reset_index(drop=True)
            right_text = right.astype("string").fillna("<NA>").reset_index(drop=True)
            if not left_text.equals(right_text):
                return False
    return True


def save_new_or_equivalent(frame, file_path):
    if file_path.is_file():
        existing = pd.read_csv(file_path, low_memory=False)
        if not frames_equivalent(frame.reset_index(drop=True), existing):
            raise FileExistsError(
                f"Existing artifact truly differs and was not overwritten: {file_path.name}"
            )
        return "already equivalent"
    frame.to_csv(file_path, index=False)
    return "created"


final_artifacts = {
    "full_cohort_fold_assignments.csv": full_cohort_folds,
    "full_cohort_fold_balance.csv": fold_balance,
    "reproduction_check.csv": reproduction_check,
    "selector_partition_summary.csv": full_cohort["selector_summary"],
    "engineering_partition_summary.csv": full_cohort["engineering_summary"],
    "screening_candidate_pool.csv": full_cohort["screening_pool"],
    "family_screen_partition_summary.csv": full_cohort["family_summary"],
    "dummy_benchmark_partition_summary.csv": full_cohort["dummy_summary"],
    "focused_tuning_partition_summary.csv": full_cohort["tuning_summary"],
    "full_cohort_tuned_winners.csv": full_cohort_winners,
    "full_cohort_selected_pipelines.csv": final_selected_pipelines,
    "full_cohort_fill_values.csv": final_fill_values,
    "model_save_status.csv": model_save_status,
    "model_reload_check.csv": model_reload_check,
    "full_cohort_validation.csv": full_cohort_validation,
}
save_rows = []
for filename, frame in final_artifacts.items():
    status = save_new_or_equivalent(frame, output_directory / filename)
    save_rows.append({"artifact": filename, "status": status, "rows": len(frame)})

display(pd.DataFrame(save_rows))
print(
    "Full-cohort direct and two-stage artifacts are saved under "
    f"{model_directory.relative_to(project_root)}. They are not performance estimates."
)

,artifact,status,rows
0,full_cohort_fold_assignments.csv,created,1040
1,full_cohort_fold_balance.csv,created,15
2,reproduction_check.csv,created,3
3,selector_partition_summary.csv,created,24
4,engineering_partition_summary.csv,created,21
5,screening_candidate_pool.csv,created,45
6,family_screen_partition_summary.csv,created,54
7,dummy_benchmark_partition_summary.csv,created,3
8,focused_tuning_partition_summary.csv,created,72
9,full_cohort_tuned_winners.csv,created,3


Full-cohort direct and two-stage artifacts are saved under results/final_pipeline/06_final_fit_and_performance_summary/09_full_cohort_fit/models. They are not performance estimates.


## Interpretation boundary

The saved direct and two-stage pipelines are the deployable and explanation artifacts for the locked final procedure. The full-cohort selection used every patient, so its selection scores and training-set predictions are optimistic and must not be reported as performance. Report Notebook 07's 20 paired outer evaluations instead, with Notebook 08's sensitivities. A publication or patient-use model still requires professor review and independent validation.